# Advanced & Ultra Hopfield Models Exploration

This notebook demonstrates the capabilities of the **Improved** (Phase 1), **Advanced** (Phase 2) and **Ultra** (Phase 3) Hopfield models for solving the Shortest Path Problem.

---

## key Features Comparison

| Feature | Improved (Base) | Advanced | Ultra |
|---------|-----------------|----------|-------|
| **Target** | Reliable Production | Large/Sparse Graphs | Performance/Scale |
| **Memory** | $O(N^2)$ | $O(E)$ (Sparse) | $O(N^2)$ (Optimized) |
| **Training** | Zero-shotted | Zero-shotted | Zero-shotted |
| **Acceleration** | CPU/Basic GPU | CPU/GPU | **Full GPU (XLA)** |
| **Heuristic** | None | Beam Search | **A* + Coordinates** |
| **Caching** | Basic | None | **Query Caching** |

# Improved Energy Function

The **Improved** and **Advanced** models use a sophisticated energy function designed to solve the Shortest Path Problem more reliably than the classic formulation. Instead of simple row/column sums, it uses **Flow Conservation**.

The energy function $E$ determines the quality of the solution:

$$
E = \mu_1 F_{cost} + \mu_2 F_{flow} + \mu_3 F_{binary} + \mu_4 F_{connect} + \mu_5 F_{sparse}
$$

### 1. Flow Conservation ($F_{flow}$)
Ensures that the path is valid by checking the net flow at each node:
$$
\sum_{k=1}^{N} \left( \sum_{j} x_{kj} - \sum_{i} x_{ik} - D_k \right)^2
$$
Where $D_k$ is the expected net flow:
- **Source**: $+1$ (Out > In)
- **Destination**: $-1$ (In > Out)
- **Intermediate**: $0$ (In == Out)

### 2. Path Cost ($F_{cost}$)
Minimizes the total weight of the selected edges:
$$
\sum_{i,j} C_{ij} x_{ij}
$$

### 3. Binary Constraint ($F_{binary}$)
Forces edge variables to be 0 or 1:
$$
\sum_{i,j} x_{ij} (1 - x_{ij})
$$

### 4. Connectivity Penalty ($F_{connect}$)
Ensures a valid route exists from Source to Destination by calculating reachability (using matrix powers).

### 5. Sparsity Penalty ($F_{sparse}$)
A regularization term to discourage activating unnecessary edges.

In [1]:
import os
import tensorflow as tf
# Disable GPU using set_visible_devices for stability
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        tf.config.set_visible_devices([], 'GPU')
    except RuntimeError:
        pass

import sys
import time
import numpy as np
import pandas as pd
import tensorflow as tf

# Add src to path
sys.path.append('..')

from src.train_model_improved import ImprovedHopfieldModel
from src.train_model_advanced import AdvancedHopfieldModel, calculate_cost_matrix
from src.train_model_ultra import create_ultra_model

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

2026-01-26 21:27:53.570258: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-26 21:27:53.895771: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769459274.015673   10037 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769459274.053524   10037 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-26 21:27:54.349456: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

TensorFlow Version: 2.18.0
GPU Available: True


W0000 00:00:1769459278.524702   10037 gpu_device.cc:2433] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


## 1. Load Data
We will use the standard synthetic network.

In [2]:
DATA_PATH = '../data/synthetic/synthetic_network.csv'

if not os.path.exists(DATA_PATH):
    print(f"Error: {DATA_PATH} not found.")
else:
    print("Data found.")
    
    # Quick peek
    df = pd.read_csv(DATA_PATH)
    print(f"Nodes: {len(set(df['origin']).union(set(df['destination'])))}")
    print(f"Edges: {len(df)}")

Data found.
Nodes: 50
Edges: 1000


## 2. Improved Model (Phase 1)
The Improved model serves as the reliable baseline, suitable for most datasets.

In [3]:
# Load and Preprocess
cost_matrix, node_mapping = calculate_cost_matrix(DATA_PATH)
n = len(cost_matrix)

# Normalize for neural network stability
cost_norm = (cost_matrix - cost_matrix.min()) / (cost_matrix.max() - cost_matrix.min())

# Initialize Improved Model
model_imp = ImprovedHopfieldModel(n, cost_norm)
model_imp.set_cost_matrix(cost_matrix)
model_imp.compile(optimizer='adam')

print("Improved Model Initialized")

Improved Model Initialized


In [4]:
SOURCE, DEST = 0, 9
start_time = time.time()
path_imp = model_imp.predict(
    source=SOURCE, 
    destination=DEST, 
    num_restarts=3, 
    validate=True
)
duration = time.time() - start_time
print(f"Path: {path_imp}")
print(f"Duration: {duration:.4f}s")

INFO:src.train_model_improved:Restart 1/3
INFO:src.train_model_improved:Optimizing path from 0 to 9
INFO:src.train_model_improved:Iteration 0, Energy: 4.7006, Temp: 1.000
INFO:src.train_model_improved:Iteration 50, Energy: 0.4459, Temp: 0.833
INFO:src.train_model_improved:Iteration 100, Energy: 0.2613, Temp: 0.667
INFO:src.train_model_improved:Iteration 150, Energy: 0.1139, Temp: 0.500
INFO:src.train_model_improved:Iteration 200, Energy: 0.0311, Temp: 0.333
INFO:src.train_model_improved:Iteration 250, Energy: 0.0032, Temp: 0.167
INFO:src.train_model_improved:Found path with cost: 1927.00
INFO:src.train_model_improved:Restart 2/3
INFO:src.train_model_improved:Optimizing path from 0 to 9
INFO:src.train_model_improved:Iteration 0, Energy: 4.6831, Temp: 1.000
INFO:src.train_model_improved:Iteration 50, Energy: 0.4467, Temp: 0.833
INFO:src.train_model_improved:Iteration 100, Energy: 0.2611, Temp: 0.667
INFO:src.train_model_improved:Iteration 150, Energy: 0.1128, Temp: 0.500
INFO:src.train_m

Path: [np.int64(0), np.int64(43), np.int64(2), np.int64(6), np.int64(5), 9]
Duration: 62.1728s


## 3. Advanced Model (Phase 2)

The Advanced model introduces **Sparse Tensor** support and **Beam Search** for better path extraction in complex graphs.

In [5]:
# Initialize Advanced Model
# use_sparse=True switches to O(E) memory mode
model_adv = AdvancedHopfieldModel(n, cost_norm, use_sparse=False)
model_adv.set_cost_matrix(cost_matrix)
model_adv.compile(optimizer='adam') # Just to initialize weights

print("Advanced Model Initialized")

INFO:src.train_model_advanced:Adaptive weights: μ1=1.00, μ2=20.00, μ3=20.00
INFO:src.train_model_advanced:Graph density: 1.000, Avg degree: 50.0


Advanced Model Initialized


In [6]:
# Test Prediction with Beam Search
start_time = time.time()
path_adv = model_adv.predict(
    source=SOURCE, 
    destination=DEST, 
    num_restarts=3,       # Parallel attempts
    validate=True,        # Ensure validity with fallback
    use_beam_search=True  # Use smart extraction
)
duration = time.time() - start_time

print(f"Path: {path_adv}")
print(f"Duration: {duration:.4f}s")

INFO:src.train_model_advanced:Restart 1/3
INFO:src.train_model_advanced:Optimizing path from 0 to 9
INFO:src.train_model_advanced:Restart 2/3
INFO:src.train_model_advanced:Optimizing path from 0 to 9
INFO:src.train_model_advanced:Restart 3/3
INFO:src.train_model_advanced:Optimizing path from 0 to 9


Path: [np.int64(0), np.int64(43), np.int64(2), np.int64(6), np.int64(5), 9]
Duration: 0.2641s


## 4. Ultra Model (Phase 3)

The Ultra model focuses on extreme performance using **XLA/GPU compilation** and **A* Heuristics**.
To use A*, we need spatial coordinates. Since our synthetic data is just a graph, we will generate synthetic coordinates for demonstration.

In [7]:
# Generate synthetic 2D coordinates for A* heuristic
# In a real scenario, these would be lat/long or x/y positions
np.random.seed(42)
coordinates = np.random.rand(n, 2) * 100  # Scale to 100x100 area

# Create Ultra Model
# This automatically enables XLA compilation
model_ultra, _, _ = create_ultra_model(DATA_PATH, coordinates=coordinates)

print("Ultra Model Initialized with Coordinates")

INFO:src.train_model_ultra:Ultra model created: 50 nodes, GPU acceleration enabled


Ultra Model Initialized with Coordinates


In [ ]:
# Test Prediction - First Run (Compilation Overhead)
print("RUN 1: First execution (includes XLA compilation time)...")
start_time = time.time()
path_ultra = model_ultra.predict(
    source=SOURCE, 
    destination=DEST, 
    num_restarts=2,
    use_cache=True
)
duration_1 = time.time() - start_time
print(f"Path: {path_ultra}")
print(f"Duration 1: {duration_1:.4f}s")

# Test Prediction - Second Run (Cached/Compiled)
print("\nRUN 2: Cached execution...")
start_time = time.time()
path_ultra_cached = model_ultra.predict(
    source=SOURCE, 
    destination=DEST, 
    num_restarts=2,
    use_cache=True
)
duration_2 = time.time() - start_time
print(f"Path: {path_ultra_cached}")
print(f"Duration 2: {duration_2:.4f}s")
print(f"Speedup: {duration_1 / duration_2:.1f}x")

RUN 1: First execution (includes XLA compilation time)...


## 5. Benchmark Comparison

Let's compare them on a longer path.

In [ ]:
# Pick a distant pair if possible, or just random
S_LONG, D_LONG = 0, n-1

print(f"Benchmarking Path {S_LONG} -> {D_LONG}")

# Improved
t0 = time.time()
p_imp = model_imp.predict(S_LONG, D_LONG, num_restarts=1)
t_imp = time.time() - t0
print(f"Improved: {t_imp:.4f}s | Path len: {len(p_imp)}")

# Advanced
t0 = time.time()
p_adv = model_adv.predict(S_LONG, D_LONG, num_restarts=1)
t_adv = time.time() - t0
print(f"Advanced: {t_adv:.4f}s | Path len: {len(p_adv)}")

# Ultra
t0 = time.time()
p_ultra = model_ultra.predict(S_LONG, D_LONG, num_restarts=1)
t_ultra = time.time() - t0
print(f"Ultra:    {t_ultra:.4f}s | Path len: {len(p_ultra)}")